# 03 Create Stratified Pilot Subset 1260

This notebook creates a representative stratified `pilot_1260` subset.

The subset preserves:

- the original dataset proportions between HAM10000 and ISIC2018
- the HAM10000 melanoma / non-melanoma label distribution
- lesion-size distribution derived from segmentation-mask coverage
- border-touch status derived from segmentation masks where available

Output:

```text
data/pilot_1260/pilot_1260_strat.csv
```

Note: ISIC2018 Task 1 provides lesion masks but no diagnostic labels. HAM10000 diagnostic labels are therefore used only for the HAM10000 subset.


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

# ============================================================
# Paths and settings


BASE_DIR = Path("..").resolve()

DATA_DIR = BASE_DIR / "data"

HAM_CSV = DATA_DIR / "preprocessed_manifests" / "ham10000_preprocessed.csv"
ISIC_CSV = DATA_DIR / "preprocessed_manifests" / "isic2018_preprocessed.csv"

RUN_NAME = "pilot_1260_strat"
PILOT_DIR = DATA_DIR /RUN_NAME
PILOT_DIR.mkdir(parents=True, exist_ok=True)

PILOT_CSV = PILOT_DIR / f"{RUN_NAME}.csv"

RANDOM_STATE = 42
TOTAL_TARGET = 1260
ID_COL = "stem"

print(f"BASE_DIR:   {BASE_DIR}")
print(f"DATA_DIR:   {DATA_DIR}")
print(f"RUN_NAME:   {RUN_NAME}")
print(f"PILOT_DIR:  {PILOT_DIR}")
print(f"PILOT_CSV:  {PILOT_CSV}")

In [ ]:
# ============================================================
# Load manifests


ham_df = pd.read_csv(HAM_CSV)
isic_df = pd.read_csv(ISIC_CSV)

ham_df["dataset"] = "HAM10000"
isic_df["dataset"] = "ISIC2018"

assert ID_COL in ham_df.columns, f"Missing {ID_COL} in HAM manifest"
assert ID_COL in isic_df.columns, f"Missing {ID_COL} in ISIC manifest"

print("HAM records:", len(ham_df))
print("ISIC records:", len(isic_df))
print("Total records:", len(ham_df) + len(isic_df))

print("\nHAM columns:")
print(ham_df.columns.tolist())

print("\nISIC columns:")
print(isic_df.columns.tolist())

In [ ]:
print((ham_df["mask_pixels_x"] - ham_df["mask_pixels_y"]).abs().sum())
print((ham_df["mask_coverage_x"] - ham_df["mask_coverage_y"]).abs().sum())

# ============================================================
# Dropping duplicates columns
ham_df["mask_pixels"] = ham_df["mask_pixels_x"]
ham_df["mask_coverage"] = ham_df["mask_coverage_x"]

ham_df.drop(
    columns=[
        "mask_pixels_x",
        "mask_pixels_y",
        "mask_coverage_x",
        "mask_coverage_y"
    ],
    inplace=True
)


isic_df["mask_pixels"] = isic_df["mask_pixels_x"]
isic_df["mask_coverage"] = isic_df["mask_coverage_x"]

isic_df.drop(
    columns=[
        "mask_pixels_x",
        "mask_pixels_y",
        "mask_coverage_x",
        "mask_coverage_y"
    ],
    inplace=True
)


In [ ]:
# ============================================================
# Helper functions


def classify_mask_size(coverage: float) -> str:
    """
    Classify lesion size from mask coverage.

    coverage = mask_pixels / total_image_pixels
    """
    if pd.isna(coverage):
        return "unknown"
    if coverage < 0.01:
        return "very_tiny"
    elif coverage < 0.02:
        return "tiny"
    elif coverage < 0.05:
        return "small"
    else:
        return "normal"


def _read_mask(mask_path: Path) -> np.ndarray:
    """
    Read a mask image as a binary numpy array.
    """
    arr = np.array(Image.open(mask_path).convert("L"))
    return arr > 0


def _resolve_path(value, base_dir: Path) -> Path | None:
    """
    Resolve a file path stored in a manifest.

    Handles absolute paths and paths relative to BASE_DIR or DATA_DIR.
    """
    if pd.isna(value):
        return None

    p = Path(str(value))

    candidates = []
    if p.is_absolute():
        candidates.append(p)
    else:
        candidates.extend([
            base_dir / p,
            DATA_DIR / p,
            Path.cwd() / p,
        ])

    for candidate in candidates:
        if candidate.exists():
            return candidate

    return None


def ensure_mask_diagnostics(
    df: pd.DataFrame,
    dataset_name: str,
    base_dir: Path = BASE_DIR,
) -> pd.DataFrame:
    """
    Ensure that a manifest contains the mask diagnostics required for sampling:

    - mask_pixels
    - mask_area_ratio
    - mask_size_class
    - touch_top
    - touch_bottom
    - touch_left
    - touch_right
    - touch_any_border

    If the values already exist, they are preserved.
    If they are missing, the function attempts to compute them from mask paths.
    """
    df = df.copy()

    required_cols = [
        "mask_pixels",
        "mask_coverage",
        "mask_size_class",
        "touch_top",
        "touch_bottom",
        "touch_left",
        "touch_right",
        "touches_any_border",
    ]

    already_available = all(col in df.columns for col in required_cols)

    if already_available:
        print(f"{dataset_name}: mask diagnostics already available.")
        return df

    # Try to find a usable mask-path column.
    candidate_mask_cols = [
        "mask_path",
        "mask_filepath",
        "mask_file",
        "mask_full_path",
        "segmentation_path",
        "segmentation_mask_path",
        "preprocessed_mask_path",
        "resized_mask_path",
    ]

    mask_col = next((col for col in candidate_mask_cols if col in df.columns), None)

    if mask_col is None:
        missing = [col for col in required_cols if col not in df.columns]
        raise ValueError(
            f"{dataset_name}: missing mask diagnostics {missing} and no recognised mask-path column found. "
            f"Available columns: {df.columns.tolist()}"
        )

    print(f"{dataset_name}: computing mask diagnostics from column: {mask_col}")

    diagnostics = []

    for _, row in df.iterrows():
        mask_path = _resolve_path(row[mask_col], base_dir)

        if mask_path is None:
            diagnostics.append({
                "mask_pixels": np.nan,
                "mask_coverage": np.nan,
                "mask_size_class": "unknown",
                "touch_top": np.nan,
                "touch_bottom": np.nan,
                "touch_left": np.nan,
                "touch_right": np.nan,
                "touches_any_border": np.nan,
            })
            continue

        mask = _read_mask(mask_path)

        h, w = mask.shape
        mask_pixels = int(mask.sum())
        total_pixels = int(h * w)
        coverage = mask_pixels / total_pixels if total_pixels > 0 else np.nan

        touch_top = bool(mask[0, :].any())
        touch_bottom = bool(mask[-1, :].any())
        touch_left = bool(mask[:, 0].any())
        touch_right = bool(mask[:, -1].any())
        touches_any_border = (touch_top or touch_bottom or touch_left or touch_right)
        mask_size_class = classify_mask_size(coverage)

        diagnostics.append({
            "mask_pixels": mask_pixels,
            "mask_coverage": coverage,
            "mask_size_class": mask_size_class,
            "touch_top": touch_top,
            "touch_bottom": touch_bottom,
            "touch_left": touch_left,
            "touch_right": touch_right,
            "touches_any_border": touches_any_border,
        })

    diag_df = pd.DataFrame(diagnostics)

    for col in required_cols:
        if col not in df.columns:
            df[col] = diag_df[col].values

    print(f"{dataset_name}: computed mask diagnostics.")
    print("Missing mask diagnostics rows:", int(df["mask_size_class"].eq("unknown").sum()))

    return df


def allocate_proportional_counts(counts: pd.Series, target_n: int) -> pd.Series:
    """
    Allocate target_n samples proportionally to observed stratum counts.

    Uses largest-remainder rounding so that the final total is exactly target_n.
    """
    counts = counts.astype(int)
    raw = counts / counts.sum() * target_n

    allocation = np.floor(raw).astype(int)
    remainder = target_n - allocation.sum()

    if remainder > 0:
        fractional = (raw - allocation).sort_values(ascending=False)
        for idx in fractional.index[:remainder]:
            allocation.loc[idx] += 1

    elif remainder < 0:
        fractional = (raw - allocation).sort_values(ascending=True)
        removable = fractional[allocation.loc[fractional.index] > 0]
        for idx in removable.index[:abs(remainder)]:
            allocation.loc[idx] -= 1

    assert allocation.sum() == target_n, (
        f"Allocation error: expected {target_n}, got {allocation.sum()}"
    )

    return allocation


def proportional_stratified_sample(
    df: pd.DataFrame,
    target_n: int,
    strata_cols,
    id_col: str = ID_COL,
    random_state: int = RANDOM_STATE,
) -> pd.DataFrame:
    """
    Sample target_n rows proportionally across one or more strata columns.
    """
    df = df.copy()

    if isinstance(strata_cols, str):
        strata_cols = [strata_cols]

    assert id_col in df.columns, f"Missing ID column: {id_col}"

    for col in strata_cols:
        assert col in df.columns, f"Missing stratification column: {col}"

    df["_stratum"] = (
        df[strata_cols]
        .astype(str)
        .agg("__".join, axis=1)
    )

    counts = df["_stratum"].value_counts()
    allocation = allocate_proportional_counts(counts, target_n)

    parts = []

    for stratum, n in allocation.items():
        if n == 0:
            continue

        group = df[df["_stratum"] == stratum]

        if n > len(group):
            n = len(group)

        parts.append(
            group.sample(
                n=n,
                random_state=random_state,
                replace=False,
            )
        )

    sample = (
        pd.concat(parts)
        .drop(columns=["_stratum"])
        .sample(frac=1, random_state=random_state)
        .reset_index(drop=True)
    )

    assert len(sample) == target_n, (
        f"Sampling error: expected {target_n}, got {len(sample)}"
    )

    return sample


In [ ]:
# ============================================================
# Ensure mask diagnostics are available for both datasets

# This makes HAM10000 and ISIC2018 consistent for lesion-size
# and border-touch analyses.

ham_df = ensure_mask_diagnostics(ham_df, "HAM10000", BASE_DIR)
isic_df = ensure_mask_diagnostics(isic_df, "ISIC2018", BASE_DIR)

required_mask_cols = ["mask_size_class", "touches_any_border"]
for name, df in [("HAM10000", ham_df), ("ISIC2018", isic_df)]:
    for col in required_mask_cols:
        assert col in df.columns, f"{name} manifest missing required column: {col}"

print("\nHAM mask size distribution:")
display(ham_df["mask_size_class"].value_counts(dropna=False))
print("\nHAM border-touch distribution:")
display(ham_df["touches_any_border"].value_counts(dropna=False))

print("\nISIC mask size distribution:")
display(isic_df["mask_size_class"].value_counts(dropna=False))
print("\nISIC border-touch distribution:")
display(isic_df["touches_any_border"].value_counts(dropna=False))


In [ ]:
# ============================================================
# Dataset-level proportional allocation


dataset_counts = pd.Series({
    "HAM10000": len(ham_df),
    "ISIC2018": len(isic_df),
})

dataset_allocation = allocate_proportional_counts(
    dataset_counts,
    TOTAL_TARGET
)

N_HAM = int(dataset_allocation["HAM10000"])
N_ISIC = int(dataset_allocation["ISIC2018"])

print("Dataset allocation:")
display(dataset_allocation)

print(f"HAM target:  {N_HAM}")
print(f"ISIC target: {N_ISIC}")
print(f"Total:       {N_HAM + N_ISIC}")

In [ ]:
# ============================================================
# HAM10000: representative sampling by label, lesion size and border-touch status


required_ham_cols = ["label", "mask_size_class", "touches_any_border"]
for col in required_ham_cols:
    assert col in ham_df.columns, f"HAM manifest must contain column: {col}"

ham_sample = proportional_stratified_sample(
    ham_df,
    target_n=N_HAM,
    strata_cols=["label", "mask_size_class", "touches_any_border"],
    id_col=ID_COL,
    random_state=RANDOM_STATE,
)

print("HAM sample:", len(ham_sample))

print("\nHAM label counts:")
display(ham_sample["label"].value_counts().sort_index())

print("\nHAM label percentages:")
display((ham_sample["label"].value_counts(normalize=True).sort_index() * 100).round(2))

print("\nHAM mask size counts:")
display(ham_sample["mask_size_class"].value_counts())

print("\nHAM mask size percentages:")
display((ham_sample["mask_size_class"].value_counts(normalize=True) * 100).round(2))

print("\nHAM border-touch counts:")
display(ham_sample["touches_any_border"].value_counts())

print("\nHAM border-touch percentages:")
display((ham_sample["touches_any_border"].value_counts(normalize=True) * 100).round(2))


In [ ]:
# ============================================================
# ISIC2018: representative sampling by mask diagnostics


required_isic_cols = ["mask_size_class", "touches_any_border"]
for col in required_isic_cols:
    assert col in isic_df.columns, f"ISIC manifest missing required column: {col}"

isic_sample = proportional_stratified_sample(
    isic_df,
    target_n=N_ISIC,
    strata_cols=["mask_size_class", "touches_any_border"],
    id_col=ID_COL,
    random_state=RANDOM_STATE,
)

print("ISIC sample:", len(isic_sample))

print("\nISIC mask size counts:")
display(isic_sample["mask_size_class"].value_counts())

print("\nISIC mask size percentages:")
display((isic_sample["mask_size_class"].value_counts(normalize=True) * 100).round(2))

print("\nISIC border-touch counts:")
display(isic_sample["touches_any_border"].value_counts())

print("\nISIC border-touch percentages:")
display((isic_sample["touches_any_border"].value_counts(normalize=True) * 100).round(2))

In [ ]:
# ============================================================
# Merge and save pilot subset

pilot_df = (
    pd.concat([ham_sample, isic_sample])
    .sample(frac=1, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)

assert len(pilot_df) == TOTAL_TARGET, f"Expected {TOTAL_TARGET}, got {len(pilot_df)}"
assert pilot_df[ID_COL].notna().all(), f"Missing values in ID column: {ID_COL}"

pilot_df.to_csv(PILOT_CSV, index=False)

print(f"Saved: {PILOT_CSV}")
print("Total pilot records:", len(pilot_df))

print("\nDataset distribution:")
display(pilot_df["dataset"].value_counts())
display((pilot_df["dataset"].value_counts(normalize=True) * 100).round(2))

print("\nHAM label distribution:")
ham_pilot = pilot_df[pilot_df["dataset"] == "HAM10000"]
display(ham_pilot["label"].value_counts().sort_index())
display((ham_pilot["label"].value_counts(normalize=True).sort_index() * 100).round(2))

display(ham_pilot["mask_size_class"].value_counts())
display((ham_pilot["mask_size_class"].value_counts(normalize=True) * 100).round(2))
print("HAM border-touch distribution:")
display(ham_pilot["touches_any_border"].value_counts())
display((ham_pilot["touches_any_border"].value_counts(normalize=True) * 100).round(2))




print("\nISIC mask size distribution:")
isic_pilot = pilot_df[pilot_df["dataset"] == "ISIC2018"]
display(isic_pilot["mask_size_class"].value_counts())
display((isic_pilot["mask_size_class"].value_counts(normalize=True) * 100).round(2))

print("\nISIC border-touch distribution:")
display(isic_pilot["touches_any_border"].value_counts())
display((isic_pilot["touches_any_border"].value_counts(normalize=True) * 100).round(2))


In [ ]:
# ============================================================
# Optional sanity check: compare full data vs pilot distributions

print("Full dataset allocation:")
display(dataset_counts)
display((dataset_counts / dataset_counts.sum() * 100).round(2))

print("\nPilot dataset allocation:")
display(pilot_df["dataset"].value_counts())
display((pilot_df["dataset"].value_counts(normalize=True) * 100).round(2))

print("\nFull HAM label distribution:")
display(ham_df["label"].value_counts().sort_index())
display((ham_df["label"].value_counts(normalize=True).sort_index() * 100).round(2))

print("\nPilot HAM label distribution:")
display(ham_pilot["label"].value_counts().sort_index())
display((ham_pilot["label"].value_counts(normalize=True).sort_index() * 100).round(2))

print("\nFull HAM mask size distribution:")
display(ham_df["mask_size_class"].value_counts())
display((ham_df["mask_size_class"].value_counts(normalize=True) * 100).round(2))

print("\nPilot HAM mask size distribution:")
display(ham_pilot["mask_size_class"].value_counts())
display((ham_pilot["mask_size_class"].value_counts(normalize=True) * 100).round(2))

print("\nFull HAM border-touch distribution:")
display(ham_df["touches_any_border"].value_counts())
display((ham_df["touches_any_border"].value_counts(normalize=True) * 100).round(2))

print("\nPilot HAM border-touch distribution:")
display(ham_pilot["touches_any_border"].value_counts())
display((ham_pilot["touches_any_border"].value_counts(normalize=True) * 100).round(2))

print("\nFull ISIC mask size distribution:")
display(isic_df["mask_size_class"].value_counts())
display((isic_df["mask_size_class"].value_counts(normalize=True) * 100).round(2))

print("\nPilot ISIC mask size distribution:")
display(isic_pilot["mask_size_class"].value_counts())
display((isic_pilot["mask_size_class"].value_counts(normalize=True) * 100).round(2))

print("\nFull ISIC border-touch distribution:")
display(isic_df["touches_any_border"].value_counts())
display((isic_df["touches_any_border"].value_counts(normalize=True) * 100).round(2))

print("\nPilot ISIC border-touch distribution:")
display(isic_pilot["touches_any_border"].value_counts())
display((isic_pilot["touches_any_border"].value_counts(normalize=True) * 100).round(2))
